### Install requirements

In [ ]:
# install libs
!pip install pandas timm scikit-learn

### Test with no pre trained model and no pre processing

In [ ]:
import pandas as pd
from pathlib import Path
from timm.data.readers.reader import Reader

# helper function to allow the dataset to work with the timm ImageDataset object
class CsvReader(Reader):
    def __init__(self, img_dir, csv_path, ext=".jpg"):
        super().__init__()
        df = pd.read_csv(csv_path)
        self.img_dir, self.ext = Path(img_dir), ext
        self.ids = df["image_id"].astype(str).str.zfill(5).tolist()  # files are like 00001.jpg and so on
        self.class_to_idx = {c: i for i, c in enumerate(sorted(df["class"].unique()))}  # converts the classes into integer ids
        self.targets = [self.class_to_idx[c] for c in df["class"]]  # creates new column with the classes integer ids

    def __getitem__(self, i):
        # timm takes the open file handle and decodes it with PIL
        return open(self.img_dir / f"{self.ids[i]}{self.ext}", "rb"), self.targets[i]

    def __len__(self):
        return len(self.ids)

    # gets file name
    def _filename(self, index, basename=False, absolute=False):
        return str(self.img_dir / f"{self.ids[index]}{self.ext}")


In [ ]:
import time, numpy as np, torch, timm
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from timm.data import ImageDataset, resolve_data_config, create_transform

SUBSET = 8000 # No. of images used from dataset
EPOCHS = 5

reader = CsvReader("dataset/train/train", "dataset/train_labels.csv")
NUM_CLASSES = len(reader.class_to_idx)
class_names = sorted(reader.class_to_idx, key=reader.class_to_idx.get)  # classes ordered by their integer id
targets = np.array(reader.targets)  # integer label per image, used to stratify the splits

print(NUM_CLASSES, class_names)

# Use a non pre trained model to gauge what performence is with no pre processing
model = timm.create_model("resnet50", pretrained=False, num_classes=NUM_CLASSES)
cfg = resolve_data_config({}, model=model)  # use default config for the model

# creates the training and validation dataset
train_full = ImageDataset(root="train/", reader=reader, transform=create_transform(**cfg, is_training=True))
val_full = ImageDataset(root="train/", reader=reader, transform=create_transform(**cfg, is_training=False))

all_idx = np.arange(len(reader))
# cut down the dataset to the 8000 while keeping the class balance (stratifying)
all_idx, _ = train_test_split(all_idx, train_size=SUBSET, stratify=targets, random_state=42)
# split that subset to 80% train and 20% val stratified by class
tr_idx, va_idx = train_test_split(all_idx, test_size=0.2, stratify=targets[all_idx], random_state=42)
print(f"train {len(tr_idx)}, val {len(va_idx)}")

# we have to use num_workers=0 here because of the custom class we're using to make the dataset compatible with timm
# since timm will pickle the dataset object and deploy it to the workers, however that won't work in this case
train_loader = DataLoader(Subset(train_full, tr_idx), batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(Subset(val_full, va_idx), batch_size=64, num_workers=0)

# use metal on m1 for speed up training
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
model.to(device)
opt = torch.optim.AdamW(model.parameters())  # Adam with default params
loss_fn = torch.nn.CrossEntropyLoss()  # standard multi-class classification loss

# Train model
for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()  # put model into training mode
    for x, y in train_loader:
        # move to metal
        x, y = x.to(device), y.to(device)
        opt.zero_grad()  # clear the gradients left over from the previous batch
        loss = loss_fn(model(x), y)  # calc loss
        # backpropagate
        loss.backward()
        # update weights
        opt.step()

    model.eval()  # put model into inference mode for evaluation
    hits = np.zeros(NUM_CLASSES); counts = np.zeros(NUM_CLASSES)  # 2 arrays to keep track of model performance
    # disable gradient tracking, saves on memory
    with torch.no_grad():
        for x, y in val_loader:
            pred = model(x.to(device)).argmax(1).cpu()  # get highest predicted class
            for c in range(NUM_CLASSES):
                m = y == c  # true if current class in loop matches the class for this image
                counts[c] += m.sum().item() # if actual class and class in the loop are the same, add it to the classes count
                hits[c] += (pred[m] == c).sum().item() # if predicted class is correct, add it to the classes' hits
    per_class = hits / np.maximum(counts, 1)  # per-class accuracy (recall), maximum is to not divide by 0

    print(f"epoch {epoch}: last train loss {loss.item():.3f}, val acc {hits.sum()/counts.sum():.3f}, "
          f"balanced acc {per_class.mean():.3f} ({time.time()-t0:.0f}s)")

print("\nper-class val accuracy:")
for n, a, c in zip(class_names, per_class, counts):
    print(f"  {n:10s} {a:.3f}  (n={int(c)})")


### Step 0: data exploration

In [ ]:
import os, pandas as pd, numpy as np
from PIL import Image

test_dir = "dataset/test/test"; tf = sorted(os.listdir(test_dir)); print("test files:", len(tf))  # sorted so the order is the same every run

labels = pd.read_csv("dataset/train_labels.csv")

# analyse each classes' image size and colour mode to see if the model could find a way to cheat
rng = np.random.default_rng(0)  # seeded so the same 600 images are picked every run
rows = []
for i in rng.choice(len(labels), 600, replace=False):
    im = Image.open(f"dataset/train/train/{str(labels.image_id[i]).zfill(5)}.jpg") # gets image data
    rows.append((labels["class"][i], *im.size, im.mode))  # adds image size and color mode to the class
sz = pd.DataFrame(rows, columns=["cls", "w", "h", "mode"]) # convert to df
print(sz[["w","h"]].describe().round(0)); print(sz["mode"].value_counts())
print(sz.groupby("cls")[["w","h"]].mean().round(0))  # average width/height per class

# each class has a unique resolution which could lead to reward hacking, all images are RGB

In [ ]:
import hashlib

# helper function to create md5 hashes
def md5(p):
    with open(p, "rb") as f: return hashlib.md5(f.read()).hexdigest()
    
train_hashes = {i: md5(f"dataset/train/train/{str(i).zfill(5)}.jpg") for i in labels.image_id}  # image_id -> hash
test_hases = {f: md5(f"{test_dir}/{f}") for f in tf}  # test file name -> hash

# attach the hash to each image
labels["hash"] = labels.image_id.map(train_hashes)

# check for duplicates (keep=False marks every member of a duplicate group, not just the repeats)
dupes = labels[labels.duplicated("hash", keep=False)]
print(f"train images in duplicate groups: {len(dupes)} ({dupes.hash.nunique()} matches)")

# check if images that have the same hash have conflicting labels
conflict = dupes.groupby("hash")["class"].nunique(); print("groups with conflicting labels:", (conflict > 1).sum())

# check if any of the hashes appear in the test dataset
train_set = set(train_hashes.values()); print("test images also in train (exact bytes):", sum(h in train_set for h in test_hases.values()), "of", len(test_hases))

# full width/height for every training image (PIL only reads the file header for .size, so this is fairly quick)
labels["w"], labels["h_px"] = zip(*[Image.open(f"dataset/train/train/{str(i).zfill(5)}.jpg").size for i in labels.image_id])
print(labels.groupby("class")[["w","h_px"]].agg(["median","min","max"]))

In [ ]:
from sklearn.metrics import confusion_matrix

# run model eval
model.eval(); P, Y = [], []
with torch.no_grad():
    for x, y in val_loader:
        P.append(model(x.to(device)).argmax(1).cpu()); Y.append(y) # append the actual and predicted class to the arrays

# P = predictions, Y = truth labels
P, Y = torch.cat(P).numpy(), torch.cat(Y).numpy()

# create confusion matrix from results
cm = pd.DataFrame(confusion_matrix(Y, P), index=[f"true {c}" for c in class_names], columns=[c[:6] for c in class_names])
print(cm)

sp = lambda c: c.replace("Cat","").replace("Dog",""); an = lambda c: c[:3]  # seperates body part to animal
print("\nanimal accuracy:", np.mean([an(class_names[a]) == an(class_names[b]) for a, b in zip(Y, P)]).round(3)) # acc for animals
print("body part accuracy:", np.mean([sp(class_names[a]) == sp(class_names[b]) for a, b in zip(Y, P)]).round(3)) # acc for body parts